# Lab 3 — Portfolio Replica Strategy
## Part 1: Data Ingestion, EDA, Autocorrelation, Monster Index

**Sections covered:**
1. Data Ingestion
2. Exploratory Data Analysis
3. Autocorrelation Analysis
4. Monster Index Construction
5. Target Index Statistics
6. Deliverable: `target_returns` + `futures_returns`

---
## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import scipy.stats as stats
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

sns.set_theme(style="whitegrid")
sns.set_context("notebook", font_scale=1.5)

---
## 1. Data Ingestion

In [ ]:
# ── Path ──────────────────────────────────────────────────────────────────────
# On Google Colab: upload the file and set file_path accordingly, e.g.
# from google.colab import files; files.upload()
file_path = 'Dataset3_PortfolioReplicaStrategy.xlsx'

# ── Load price levels ─────────────────────────────────────────────────────────
# header=0  → row 1 of the Excel file is the column header
# index_col=0 → first column (dates) becomes the DatetimeIndex
prices = pd.read_excel(file_path, header=0, index_col=0, parse_dates=True)

# ── Human-readable labels ─────────────────────────────────────────────────────
LABEL = {
    'MXWO':     'MSCI World',
    'MXWD':     'MSCI ACWI',
    'LEGATRUU': 'Global Aggregate Bond',
    'HFRXGL':   'HFRX Hedge Fund Global',
    'RX1':  'Bund Future (10Y GER)',
    'TY1':  '10Y US Treasury Future',
    'GC1':  'Gold Future',
    'CO1':  'Brent Crude Future',
    'ES1':  'S&P 500 Future',
    'VG1':  'EuroStoxx 50 Future',
    'NQ1':  'Nasdaq 100 Future',
    'LLL1': 'MSCI EM Future',
    'TP1':  'TOPIX Future',
    'DU1':  'Schatz 2Y GER Future',
    'TU2':  '2Y US Treasury Future',
}

# ── Column groups ─────────────────────────────────────────────────────────────
TARGET_COLS  = ['MXWO', 'MXWD', 'LEGATRUU', 'HFRXGL']
FUTURES_COLS = ['RX1', 'TY1', 'GC1', 'CO1', 'ES1', 'VG1',
                'NQ1', 'LLL1', 'TP1', 'DU1', 'TU2']
ANN = 52  # weekly → annual scaling

# ── Compute weekly returns ────────────────────────────────────────────────────
# pct_change() produces NaN only on the very first row → dropna() removes it
returns = prices.pct_change().dropna()

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"Price levels  : {prices.shape[0]} rows × {prices.shape[1]} cols")
print(f"Weekly returns: {returns.shape[0]} rows × {returns.shape[1]} cols")
print(f"Date range    : {returns.index[0].date()} → {returns.index[-1].date()}")
print(f"Columns       : {returns.columns.tolist()}")
print()
display(prices.head())
display(returns.describe().round(6))

---
## 2. Exploratory Data Analysis

### 2.1 Normalized Price Series (Base = 100)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for col in TARGET_COLS:
    norm = prices[col] / prices[col].iloc[0] * 100
    ax.plot(norm, linewidth=2, label=f"{col} — {LABEL[col]}")

ax.axhline(100, color='k', linestyle='--', alpha=0.3)
ax.set_title('Target Indices: Normalized Price (Base = 100 at 2007-10-23)', fontsize=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Normalized Level', fontsize=13)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.2 Annualized Statistics

In [ ]:
def fmt_pct(x):
    return f"{x * 100:.2f}%"

stats_tbl = pd.DataFrame({
    'Ann. Return':     returns[TARGET_COLS].mean() * ANN,
    'Ann. Volatility': returns[TARGET_COLS].std() * np.sqrt(ANN),
    'Sharpe Ratio':   (returns[TARGET_COLS].mean() * ANN) /
                      (returns[TARGET_COLS].std() * np.sqrt(ANN)),
    'Max Drawdown':    returns[TARGET_COLS].apply(
                          lambda x: ((1 + x).cumprod() /
                                     (1 + x).cumprod().cummax() - 1).min()),
    'Skewness':        returns[TARGET_COLS].skew(),
    'Kurtosis':        returns[TARGET_COLS].kurtosis(),
})

for col in ['Ann. Return', 'Ann. Volatility', 'Max Drawdown']:
    stats_tbl[col] = stats_tbl[col].apply(fmt_pct)
stats_tbl['Sharpe Ratio'] = stats_tbl['Sharpe Ratio'].round(3)
stats_tbl['Skewness']     = stats_tbl['Skewness'].round(3)
stats_tbl['Kurtosis']     = stats_tbl['Kurtosis'].round(3)

print("Annualized statistics for target indices (weekly data, 2007-2021):")
display(stats_tbl)

### 2.3 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = returns[TARGET_COLS].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Target Index Weekly Returns', fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Return Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, col in zip(axes.flatten(), TARGET_COLS):
    sns.histplot(returns[col], kde=True, ax=ax)
    ax.set_title(f'Return Distribution: {col} — {LABEL[col]}', fontsize=13)
    ax.set_xlabel('Weekly Return', fontsize=11)
    ax.axvline(0, color='red', linestyle='--', alpha=0.7)
    ax.annotate(
        f"μ = {returns[col].mean():.4f}\nσ = {returns[col].std():.4f}",
        xy=(0.70, 0.85), xycoords='axes fraction',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.8),
        fontsize=10
    )

plt.suptitle('Weekly Return Distributions — Target Indices', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 2.5 Cumulative Returns

In [ ]:
cum_ret = (1 + returns[TARGET_COLS]).cumprod()

fig, ax = plt.subplots(figsize=(14, 7))
for col in TARGET_COLS:
    ax.plot(cum_ret[col], linewidth=2, label=f"{col} — {LABEL[col]}")

ax.set_title('Cumulative Returns — Target Indices (1 unit initial investment)', fontsize=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Cumulative Return', fontsize=13)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.6 Rolling 52-Week Correlation vs HFRXGL

In [ ]:
base_idx = 'HFRXGL'
rolling_window = 52

fig, ax = plt.subplots(figsize=(14, 7))
for col in [c for c in TARGET_COLS if c != base_idx]:
    roll_corr = returns[[base_idx, col]].rolling(rolling_window).corr().unstack()[base_idx][col]
    ax.plot(roll_corr, linewidth=1.8, label=f"{base_idx} vs {col} ({LABEL[col]})")

ax.axhline(0, color='k', linestyle='--', alpha=0.4)
ax.set_title(f'Rolling {rolling_window}-Week Correlation with {base_idx}', fontsize=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Correlation Coefficient', fontsize=13)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.7 QQ Plots — Weekly Returns vs Normal Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), TARGET_COLS):
    stats.probplot(returns[col].dropna(), dist='norm', plot=ax)
    ax.set_title(f'QQ Plot: {col} — {LABEL[col]}', fontsize=12)
    ax.grid(True, alpha=0.3)

plt.suptitle('QQ Plots: Weekly Returns vs Normal Distribution', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. Autocorrelation Analysis

We analyze serial dependence in weekly returns. For replication purposes, understanding autocorrelation is important:
- Equity indices (MXWO, MXWD) typically show no significant autocorrelation in weekly returns (efficient market behavior).
- **HFRXGL (hedge funds) exhibits significant autocorrelation**, especially in the 2007–2011 period, due to **stale pricing** and **illiquidity effects** typical of hedge funds — they hold less liquid assets whose prices are updated infrequently, creating artificial serial correlation. This was amplified during the 2008 financial crisis.

### 3.1 ACF and PACF

In [ ]:
MAX_LAGS = 20

fig, axes = plt.subplots(len(TARGET_COLS), 2, figsize=(18, 4 * len(TARGET_COLS)))

for i, col in enumerate(TARGET_COLS):
    r = returns[col].dropna()
    plot_acf(r,  lags=MAX_LAGS, ax=axes[i, 0],
             title=f'ACF: {col} — {LABEL[col]}',  alpha=0.05)
    plot_pacf(r, lags=MAX_LAGS, ax=axes[i, 1],
             title=f'PACF: {col} — {LABEL[col]}', alpha=0.05)
    axes[i, 0].set_xlabel('Lag'); axes[i, 0].set_ylabel('Correlation')
    axes[i, 1].set_xlabel('Lag'); axes[i, 1].set_ylabel('Partial Correlation')
    axes[i, 0].grid(True, alpha=0.3)
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 Ljung-Box Test for Autocorrelation

**H₀**: No autocorrelation up to lag k. A significant p-value (< 0.05) rejects H₀.

In [ ]:
print("Ljung-Box test (H₀: no autocorrelation up to lag k)")
print("=" * 60)

for col in TARGET_COLS:
    lb = acorr_ljungbox(returns[col].dropna(), lags=[5, 10, 15, 20])
    print(f"\n{col} — {LABEL[col]}:")
    for lag_val, (_, row) in zip([5, 10, 15, 20], lb.iterrows()):
        stars = '***' if row['lb_pvalue'] < 0.001 else \
                '** ' if row['lb_pvalue'] < 0.01 else \
                '*  ' if row['lb_pvalue'] < 0.05 else '   '
        print(f"  Lag {lag_val:2d}: Q = {row['lb_stat']:8.4f},  p = {row['lb_pvalue']:.4f}  {stars}")

print("\nSignificance: *** p<0.001  ** p<0.01  * p<0.05")

### 3.3 Volatility Clustering — ACF of Squared Returns

Significant autocorrelation in **squared returns** indicates **volatility clustering** (ARCH/GARCH effects): periods of high volatility tend to be followed by high volatility.

In [ ]:
fig, axes = plt.subplots(len(TARGET_COLS), 1, figsize=(14, 4 * len(TARGET_COLS)))

for i, col in enumerate(TARGET_COLS):
    sq_ret = returns[col].dropna() ** 2
    plot_acf(sq_ret, lags=MAX_LAGS, ax=axes[i], alpha=0.05,
             title=f'ACF of Squared Returns: {col} — {LABEL[col]} (Volatility Clustering)')
    axes[i].set_xlabel('Lag')
    axes[i].set_ylabel('Autocorrelation')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Monster Index Construction

The **Monster Index** is our replication target: a non-investable, multi-asset black-box combining:
- **50% HFRXGL** — Hedge Fund Research HFRX Global Index
- **25% MXWO** — MSCI World (global developed equities)
- **25% LEGATRUU** — Bloomberg Global Aggregate Bond Index

We will attempt to replicate this with a **long/short portfolio of liquid futures contracts**.

In [ ]:
# ── Monster Index weights ──────────────────────────────────────────────────────
MONSTER_WEIGHTS = {'HFRXGL': 0.50, 'MXWO': 0.25, 'LEGATRUU': 0.25}

# Weighted sum of weekly returns
target_returns = sum(returns[col] * w for col, w in MONSTER_WEIGHTS.items())
target_returns.name = 'Monster_Index'

# Futures returns (replication instruments)
futures_returns = returns[FUTURES_COLS].copy()

# Alignment check
assert (target_returns.index == futures_returns.index).all(), \
    "Date misalignment between target_returns and futures_returns!"

print(f"Monster Index weights : {MONSTER_WEIGHTS}")
print(f"Sum of weights        : {sum(MONSTER_WEIGHTS.values())} ✓")
print(f"target_returns  shape : {target_returns.shape}")
print(f"futures_returns shape : {futures_returns.shape}")
print(f"Date range            : {target_returns.index[0].date()} → {target_returns.index[-1].date()}")

---
## 5. Target Index Statistics

> **Note on data range**: this analysis uses the full 2007–2021 history, which includes the **2008 Global Financial Crisis**. Statistics (especially max drawdown and Sharpe ratio) will differ significantly from analyses using post-2011 data.

### 5.1 Performance Metrics

In [ ]:
cum_monster = (1 + target_returns).cumprod()
dd_monster  = 1 - cum_monster / cum_monster.cummax()

metrics = pd.DataFrame({
    'Statistic': [
        'Annualized Return', 'Annualized Volatility', 'Sharpe Ratio',
        'Max Drawdown', 'Skewness', 'Kurtosis'
    ],
    'Value': [
        fmt_pct(target_returns.mean() * ANN),
        fmt_pct(target_returns.std() * np.sqrt(ANN)),
        f"{(target_returns.mean() * ANN) / (target_returns.std() * np.sqrt(ANN)):.2f}",
        fmt_pct(dd_monster.max()),
        f"{target_returns.skew():.2f}",
        f"{target_returns.kurtosis():.2f}",
    ]
})

print("Monster Index — Performance Statistics (weekly, 2007-2021):")
display(metrics)

### 5.2 Cumulative Return & Drawdown

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Cumulative return
ax1.plot(cum_monster, color='steelblue', linewidth=2)
ax1.set_title('Monster Index: Cumulative Return', fontsize=14)
ax1.set_ylabel('Cumulative Return (1 = initial investment)')
ax1.grid(True, alpha=0.3)

# Drawdown
ax2.fill_between(dd_monster.index, -dd_monster, 0, color='firebrick', alpha=0.5)
ax2.plot(-dd_monster, color='firebrick', linewidth=1)
ax2.set_title('Monster Index: Drawdown', fontsize=14)
ax2.set_ylabel('Drawdown')
ax2.set_xlabel('Date')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 5.3 Rolling Correlation: Monster Index vs Futures

In [ ]:
# Rank futures by absolute correlation with Monster Index
corr_rank = futures_returns.corrwith(target_returns).abs().sort_values(ascending=False)
top5_futures = corr_rank.index[:5].tolist()

print("Futures contracts ranked by absolute correlation with Monster Index:")
corr_full = futures_returns.corrwith(target_returns).sort_values(ascending=False)
corr_df = pd.DataFrame({'Correlation': corr_full, 'Abs Correlation': corr_full.abs()})
corr_df['Instrument'] = corr_df.index.map(LABEL)
display(corr_df.round(4))

# Rolling correlation plot
rolling_window = 52
fig, ax = plt.subplots(figsize=(14, 7))

for fut in top5_futures:
    rc = target_returns.rolling(rolling_window).corr(futures_returns[fut])
    ax.plot(rc, linewidth=1.5, label=f"{fut} — {LABEL[fut]}")

ax.axhline(0, color='k', linestyle='--', alpha=0.4)
ax.set_title(f'Rolling {rolling_window}-Week Correlation: Monster Index vs Top 5 Futures',
             fontsize=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Correlation Coefficient', fontsize=12)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.4 Scatter Matrix: Monster Index vs Top 5 Correlated Futures

In [ ]:
scatter_data = pd.concat(
    [target_returns.rename('Monster_Index'), futures_returns[top5_futures]],
    axis=1
)

g = sns.pairplot(
    scatter_data,
    kind='scatter',
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 's': 12, 'edgecolor': 'k', 'linewidth': 0.3}
)
g.fig.suptitle('Scatter Matrix: Monster Index vs Top 5 Correlated Futures',
               fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Deliverable

The following two objects are the output of this notebook, to be used by teammates for the replication model.

| Object | Type | Shape | Description |
|---|---|---|---|
| `target_returns` | `pd.Series` | (704,) | Weekly Monster Index returns, DatetimeIndex |
| `futures_returns` | `pd.DataFrame` | (704, 11) | Weekly returns of 11 futures contracts |

In [ ]:
print("=" * 55)
print("DELIVERABLE SUMMARY")
print("=" * 55)
print(f"target_returns")
print(f"  Type   : {type(target_returns).__name__}")
print(f"  Shape  : {target_returns.shape}")
print(f"  Name   : '{target_returns.name}'")
print(f"  Range  : {target_returns.index[0].date()} → {target_returns.index[-1].date()}")
print()
print(f"futures_returns")
print(f"  Type   : {type(futures_returns).__name__}")
print(f"  Shape  : {futures_returns.shape}")
print(f"  Cols   : {futures_returns.columns.tolist()}")
print(f"  Range  : {futures_returns.index[0].date()} → {futures_returns.index[-1].date()}")
print()
print(f"Indices aligned : {(target_returns.index == futures_returns.index).all()} ✓")
print("=" * 55)

# ── Optional: persist to disk for teammates ───────────────────────────────────
# target_returns.to_csv('target_returns.csv')
# futures_returns.to_csv('futures_returns.csv')
# OR use pickle for exact dtype preservation:
# import pickle
# with open('lab3_deliverables.pkl', 'wb') as f:
#     pickle.dump({'target_returns': target_returns,
#                  'futures_returns': futures_returns}, f)